In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prefixspan import PrefixSpan

In [3]:
df = pd.read_excel("Online Retail.xlsx")

print("=== 5 Baris Pertama ===")
display(df.head())

print("\n=== Dimensi Dataset ===")
n_rows, n_cols = df.shape
print(f"Jumlah baris  : {n_rows:,}")
print(f"Jumlah kolom  : {n_cols}")

print("\n=== Info Dataset ===")
df.info()

=== 5 Baris Pertama ===


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



=== Dimensi Dataset ===
Jumlah baris  : 541,909
Jumlah kolom  : 8

=== Info Dataset ===
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [4]:
cols = ["CustomerID", "InvoiceNo", "InvoiceDate", "Description"]
df_seq = df[cols].copy()

print("=== Kolom Relevan ===")
display(df_seq.head())

print("\n=== Missing Value ===")
display(df_seq.isnull().sum().to_frame("Jumlah Missing"))

=== Kolom Relevan ===


,CustomerID,InvoiceNo,InvoiceDate,Description
0,17850.0,536365,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER
1,17850.0,536365,2010-12-01 08:26:00,WHITE METAL LANTERN
2,17850.0,536365,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER
3,17850.0,536365,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE
4,17850.0,536365,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.



=== Missing Value ===


,Jumlah Missing
CustomerID,135080
InvoiceNo,0
InvoiceDate,0
Description,1454


In [5]:
n_awal = len(df_seq)

# Hapus Description kosong
df_seq = df_seq.dropna(subset=["Description"])

# Hapus transaksi dibatalkan (InvoiceNo diawali "C")
df_seq = df_seq[~df_seq["InvoiceNo"].astype(str).str.startswith("C")]

print(f"Baris sebelum : {n_awal:,}")
print(f"Baris sesudah : {len(df_seq):,}")
print(f"Baris dihapus : {n_awal - len(df_seq):,}")

Baris sebelum : 541,909
Baris sesudah : 531,167
Baris dihapus : 10,742


In [6]:
n_awal = len(df_seq)

df_seq = df_seq[df.loc[df_seq.index, "Quantity"] > 0]

print(f"Baris sebelum : {n_awal:,}")
print(f"Baris sesudah : {len(df_seq):,}")
print(f"Baris dihapus : {n_awal - len(df_seq):,}")

Baris sebelum : 531,167
Baris sesudah : 530,693
Baris dihapus : 474


In [7]:
n_awal = len(df_seq)

df_seq = df_seq.dropna(subset=["CustomerID"])

print(f"Baris sebelum : {n_awal:,}")
print(f"Baris sesudah : {len(df_seq):,}")
print(f"Baris dihapus : {n_awal - len(df_seq):,}")

Baris sebelum : 530,693
Baris sesudah : 397,924
Baris dihapus : 132,769


In [8]:
ringkasan = pd.DataFrame({
    "Sebelum": [len(df), df["InvoiceNo"].nunique(), df["CustomerID"].nunique()],
    "Sesudah": [len(df_seq), df_seq["InvoiceNo"].nunique(), df_seq["CustomerID"].nunique()],
}, index=["Jumlah Baris", "Jumlah Invoice", "Jumlah Pelanggan"])

ringkasan["Dihapus"] = ringkasan["Sebelum"] - ringkasan["Sesudah"]

display(ringkasan.style.format("{:,}"))

,Sebelum,Sesudah,Dihapus
Jumlah Baris,"541,909","397,924","143,985"
Jumlah Invoice,"25,900","18,536","7,364"
Jumlah Pelanggan,"4,372","4,339",33


In [9]:
df_seq = df_seq.sort_values(["CustomerID", "InvoiceDate"])

# Satu invoice = satu itemset (item duplikat dalam invoice dibuang)
itemsets = (
    df_seq.groupby(["CustomerID", "InvoiceNo", "InvoiceDate"])["Description"]
    .apply(lambda x: sorted(set(x.str.strip())))
    .reset_index(name="Itemset")
    .sort_values(["CustomerID", "InvoiceDate"])
    .reset_index(drop=True)
)

print(f"Jumlah itemset (invoice): {len(itemsets):,}")
display(itemsets.head())

Jumlah itemset (invoice): 18,566


,CustomerID,InvoiceNo,InvoiceDate,Itemset
0,12346.0,541431,2011-01-18 10:01:00,[MEDIUM CERAMIC TOP STORAGE JAR]
1,12347.0,537626,2010-12-07 14:57:00,"[3D DOG PICTURE PLAYING CARDS, AIRLINE BAG VIN..."
2,12347.0,542237,2011-01-26 14:30:00,"[3D DOG PICTURE PLAYING CARDS, 60 TEATIME FAIR..."
3,12347.0,549222,2011-04-07 10:43:00,"[3D SHEET OF CAT STICKERS, 3D SHEET OF DOG STI..."
4,12347.0,556201,2011-06-09 13:01:00,"[3D DOG PICTURE PLAYING CARDS, AIRLINE BAG VIN..."


In [ ]:
sequences = (
    itemsets.groupby("CustomerID")["Itemset"]
    .apply(list)
    .tolist()
)

print(f"Jumlah sequence (pelanggan): {len(sequences):,}")
print("Contoh sequence pertama (2 itemset awal):")
print(sequences[0][:2])

# PrefixSpan menerima sequence berisi item tunggal, jadi itemset diratakan
# (item dalam satu invoice tetap berurutan sesuai urutan alfabet)
sequences_flat = [[item for itemset in seq for item in itemset] for seq in sequences]

Jumlah sequence (pelanggan): 4,339
Contoh sequence pertama (2 itemset awal):
[['MEDIUM CERAMIC TOP STORAGE JAR']]


In [17]:
import time

n_seq = len(sequences_flat)
nilai_support = [0.05, 0.03, 0.02, 0.01]
maxlen = 3   # batasi panjang pola agar nilai kecil tetap bisa diproses; set None untuk tanpa batas

hasil_eksperimen = []
for ms in nilai_support:
    min_count = int(ms * n_seq)
    ps = PrefixSpan(sequences_flat)
    if maxlen:
        ps.maxlen = maxlen

    mulai = time.time()
    pola = ps.frequent(min_count)
    durasi = time.time() - mulai

    hasil_eksperimen.append({
        "Min Support": ms,
        "Min Pelanggan": min_count,
        "Jumlah Pola": len(pola),
        "Waktu (detik)": round(durasi, 1),
    })

tabel_eksperimen = pd.DataFrame(hasil_eksperimen)
display(tabel_eksperimen)



,Min Support,Min Pelanggan,Jumlah Pola,Waktu (detik)
0,0.05,216,341,6.0
1,0.03,130,1584,18.2
2,0.02,86,6832,44.0
3,0.01,43,83593,157.6


In [18]:
min_support = 0.02
min_count = int(min_support * len(sequences_flat))   # 86 pelanggan

ps = PrefixSpan(sequences_flat)
ps.maxlen = 3

patterns = ps.frequent(min_count)

print(f"Min support : {min_support:.0%} ({min_count} pelanggan)")
print(f"Jumlah pola : {len(patterns):,}")

Min support : 2% (86 pelanggan)
Jumlah pola : 6,832


In [20]:
n_seq = len(sequences_flat)
support_dict = {tuple(p): s for s, p in patterns}

rows = []
for s, p in patterns:
    if len(p) >= 2:   # pola 1 item tidak punya confidence
        conf = s / support_dict[tuple(p[:-1])]
        rows.append({
            "Pola": " -> ".join(p),
            "Panjang": len(p),
            "Support": s,
            "Support (%)": round(s / n_seq * 100, 2),
            "Confidence (%)": round(conf * 100, 2),
        })

hasil = (pd.DataFrame(rows)
         .sort_values(["Support", "Confidence (%)"], ascending=False)
         .reset_index(drop=True))

print(f"Jumlah pola berurutan (panjang >= 2): {len(hasil):,}")
display(hasil.head(20))

Jumlah pola berurutan (panjang >= 2): 5,776


,Pola,Panjang,Support,Support (%),Confidence (%)
0,WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HA...,2,405,9.33,47.31
1,REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND ...,2,330,7.61,37.46
2,PAPER CHAIN KIT 50'S CHRISTMAS -> PAPER CHAIN ...,2,321,7.40,52.37
3,GREEN REGENCY TEACUP AND SAUCER -> ROSES REGEN...,2,318,7.33,82.17
4,JUMBO BAG RED RETROSPOT -> JUMBO BAG RED RETRO...,2,312,7.19,49.13
5,REGENCY CAKESTAND 3 TIER -> ROSES REGENCY TEAC...,2,299,6.89,33.94
6,GREEN REGENCY TEACUP AND SAUCER -> PINK REGENC...,2,294,6.78,75.97
7,RED HANGING HEART T-LIGHT HOLDER -> WHITE HANG...,2,293,6.75,76.10
8,LUNCH BAG CARS BLUE -> LUNCH BAG RED RETROSPOT,2,293,6.75,61.17
9,HEART OF WICKER LARGE -> HEART OF WICKER SMALL,2,292,6.73,60.21


In [21]:
pd.set_option("display.max_colwidth", None)

top3 = hasil.head(3)
display(top3)

,Pola,Panjang,Support,Support (%),Confidence (%)
0,WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HANGING HEART T-LIGHT HOLDER,2,405,9.33,47.31
1,REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND 3 TIER,2,330,7.61,37.46
2,PAPER CHAIN KIT 50'S CHRISTMAS -> PAPER CHAIN KIT VINTAGE CHRISTMAS,2,321,7.40,52.37


**Pola 1 dan 2: pembelian berulang (repeat purchase).** Produk yang sama dibeli lagi di transaksi berikutnya. Karena item dalam satu invoice sudah dibuang duplikatnya, pengulangan ini pasti terjadi di invoice yang berbeda, jadi benar-benar pembelian ulang.
- **White Hanging Heart T-Light Holder**: 9,33% pelanggan membelinya lebih dari sekali, dan sekitar separuh pembelinya kembali membeli. Ini produk andalan dengan perputaran tinggi, sehingga stoknya perlu dijaga dan pelanggan bisa diberi penawaran restock.
- **Regency Cakestand 3 Tier**: pola serupa dengan confidence lebih rendah (37,46%). Pembeliannya kemungkinan berupa pengisian stok oleh pelanggan grosir atau tambahan untuk acara berikutnya.

**Pola 3: pembelian lintas varian (cross-selling antar produk sejenis).** Pelanggan yang membeli Paper Chain Kit 50's Christmas cenderung membeli Paper Chain Kit Vintage Christmas, dengan confidence tertinggi (52,37%). Artinya lebih dari separuh pembeli varian pertama juga membeli varian kedua. Tindakan bisnisnya bisa berupa bundling kedua varian, rekomendasi varian lain saat checkout, dan persiapan stok menjelang musim Natal karena produknya musiman.